In [1]:
import pandas as pd
import numpy as np
pd.options.plotting.backend = "plotly"

X_train = pd.read_csv('../../data/X_train.csv', sep = ',', index_col=0)
X_test = pd.read_csv('../../data/X_test.csv', sep = ',', index_col=0)
y_train = pd.read_csv('../../data/y_train.csv', sep = ',', index_col=0)
df = pd.merge(X_train, y_train, left_index=True, right_index=True)
X_test["MathScore"] = None
df = pd.concat([df, X_test], ignore_index=False)

final_cols = []
to_rs = []
to_oh = []
to_te = []

/tmp/ipykernel_46574/2031058060.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, X_test], ignore_index=False)


In [2]:
attempts = df.groupby(["CNTSTUID", "Year"]).size()

cum = attempts.groupby(level=0).cumsum()
time_per_year = (cum - attempts + 1).rename("Attempts")
df = df.join(time_per_year, on=["CNTSTUID", "Year"])
to_rs.append("Attempts")
to_te.append("CNTSTUID")
to_oh.append("Year")

In [3]:
#Basic FillNa 0
fillna_0=["Option_CT",
"Option_FL",
"Option_ICTQ",
"Option_WBQ",
"Option_PQ",
"Option_TQ",
"Option_UH",
"EFFORT1",
"EFFORT2",
"IMMIG",
"MISSSC"
]
df[fillna_0]=df[fillna_0].fillna(0)

final_cols += ["Option_CT",
"Option_FL",
"Option_ICTQ",
"Option_WBQ",
"Option_PQ",
"Option_TQ",
"Option_UH",
"MISSSC"
]
to_rs += ["EFFORT1",
"EFFORT2",
"IMMIG",
]

In [4]:
cols = ["ST003D02T", "ST003D03T", "ST004D01T"]

for c in cols:
    mode_val = df[c].mode().iloc[0]   # valeur la plus fréquente
    df[c] = df[c].fillna(mode_val)
    
to_rs += ["ST003D02T","ST003D03T"]
df["ST004D01T"] = df["ST004D01T"] -1 
final_cols.append("ST004D01T")

In [6]:
isco_conv = pd.read_csv("../../data/isco08_isei08_full_with_nearest.csv", sep=",",index_col=0)
isco_dict = isco_conv.to_dict()["isei08"]
ocods=["OCOD1","OCOD2","OCOD3"]
for c in ocods:
    df[c]=df[c].map(isco_dict)


In [7]:
ocodsmean = df.groupby("STRATUM")[ocods].mean().mean()
fill_ocod = df.groupby("CNTSCHID")[ocods].transform("mean")
for c in ocods:
    df[c] = df[c].fillna(fill_ocod[c]).fillna(ocodsmean[c])

to_rs+=ocods

In [8]:
df["AGE"]=df["AGE"].fillna(df["Year"]-df["ST003D03T"])
df["AGE"]=df["AGE"] - df.groupby(["Year","CNTSCHID"])["AGE"].transform("mean")

to_rs.append("AGE")

In [9]:
grade_fill_1=df.groupby(["CNTSCHID","Year"])["GRADE"].transform("median")
grade_fill_2=df.groupby(["Year"])["GRADE"].transform("median")
grade_fill_3=df["GRADE"].mean()

df["GRADE"]=df["GRADE"].fillna(grade_fill_1).fillna(grade_fill_2).fillna(grade_fill_3)

to_rs.append("GRADE")

In [10]:
COBN_S_fil_1 = df.groupby(["CNTSCHID", "Year"])["COBN_S"] \
    .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else s.mean())

COBN_S_fil_2 = df.groupby("CNTSCHID")["COBN_S"] \
    .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else s.mean())

COBN_S_fil_3 = df["COBN_S"].mean()

# Chaîne d’imputation
df["COBN_S"] = (
    df["COBN_S"]
      .fillna(COBN_S_fil_1)
      .fillna(COBN_S_fil_2)
      .fillna(COBN_S_fil_3)
)

to_te.append("COBN_S")


In [11]:
mathease_fil_1 = df.groupby(["CNTSCHID", "ST004D01T"])["MATHEASE"] \
    .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else s.mean())

mathease_fil_2 = df.groupby("CNTSCHID")["MATHEASE"] \
    .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else s.mean())

mathease_fil_3 = df["MATHEASE"].mean()

# Chaîne d’imputation
df["MATHEASE"] = (
    df["MATHEASE"]
      .fillna(mathease_fil_1)
      .fillna(mathease_fil_2)
      .fillna(mathease_fil_3)
)

final_cols.append("MATHEASE")


In [12]:
#Fin Fill na, création de feature

In [13]:
df["LANGTEST_PAQ"]=(~df["LANGTEST_PAQ"].isna())

In [14]:
df["DIFFERENT"] =(df["LANGTEST_COG"] != df.groupby(["CNTSCHID"])["LANGTEST_COG"]
.transform(lambda s: s.mode().iloc[0])).astype(int)

final_cols.append("DIFFERENT")

In [15]:
df["ST001D01T"] = df["ST001D01T"].isin([96, 97, 98, 99]).astype(int)
final_cols.append("ST001D01T")

In [16]:
df["ADMINMODE"] = (df["ADMINMODE"]-1)
final_cols.append("ADMINMODE")

In [18]:
stratum_conv=pd.read_excel("../../data/STRATUM_embedding.xlsx",index_col=0)
stratum_conv_dict = stratum_conv.to_dict()["Cluster_ID"]
stratum_conv_dict

{'ALB01': 0,
 'ALB02': 3,
 'ALB03': 0,
 'ALB04': 3,
 'ALB05': 0,
 'ALB06': 3,
 'ALB07': 3,
 'ALB08': 0,
 'ALB09': 3,
 'ALB10': 0,
 'ALB11': 3,
 'ALB97': 1,
 'ARE01': 3,
 'ARE05': 3,
 'ARE06': 3,
 'ARE07': 3,
 'ARE08': 3,
 'ARE09': 3,
 'ARE10': 3,
 'ARE11': 3,
 'ARE15': 3,
 'ARE16': 3,
 'ARE17': 3,
 'ARE18': 3,
 'ARE19': 3,
 'ARE20': 3,
 'ARE21': 3,
 'ARE25': 3,
 'ARE26': 3,
 'ARE27': 3,
 'ARE28': 3,
 'ARE29': 3,
 'ARE30': 3,
 'ARE31': 3,
 'ARE35': 3,
 'ARE36': 3,
 'ARE37': 3,
 'ARE38': 3,
 'ARE39': 3,
 'ARE40': 3,
 'ARE41': 4,
 'ARE45': 3,
 'ARE46': 3,
 'ARE47': 3,
 'ARE49': 3,
 'ARE50': 3,
 'ARE51': 4,
 'ARE55': 3,
 'ARE56': 3,
 'ARE57': 3,
 'ARE58': 3,
 'ARE59': 3,
 'ARE60': 3,
 'ARE61': 4,
 'ARE65': 3,
 'ARE66': 3,
 'ARE67': 3,
 'ARE68': 3,
 'ARE70': 3,
 'ARE97': 1,
 'ARG01': 3,
 'ARG02': 3,
 'ARG03': 3,
 'ARG04': 3,
 'ARG05': 3,
 'ARG06': 3,
 'ARG07': 3,
 'ARG08': 3,
 'ARG09': 3,
 'ARG10': 3,
 'ARG11': 3,
 'ARG12': 3,
 'ARG13': 3,
 'ARG14': 3,
 'ARG15': 3,
 'ARG16': 3,
 'ARG17': 3,

In [19]:
from collections import defaultdict

# ------------------------------------------
# 1. distance de Levenshtein
# ------------------------------------------
def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    if len(a) > len(b):
        a, b = b, a

    previous_row = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current_row = [i]
        for j, cb in enumerate(b, start=1):
            insert_cost  = current_row[j - 1] + 1
            delete_cost  = previous_row[j] + 1
            replace_cost = previous_row[j - 1] + (ca != cb)
            current_row.append(min(insert_cost, delete_cost, replace_cost))
        previous_row = current_row
    return previous_row[-1]

# ------------------------------------------
# 2. pré-calculs : clés du dict + fréquences
# ------------------------------------------
keys = list(stratum_conv_dict.keys())

def get_prefix(code: str, n: int = 3) -> str:
    s = str(code)
    return s[:n] if len(s) >= n else s

from collections import defaultdict
keys_by_prefix = defaultdict(list)
for k in keys:
    prefix = get_prefix(k)
    keys_by_prefix[prefix].append(k)

stratum_freq = df["STRATUM"].value_counts().to_dict()

# ------------------------------------------
# 3. fonction de résolution (NaN -> 5)
# ------------------------------------------
def closest_key(code, max_dist=4):
    """
    Retourne la valeur de stratum_conv_dict associée à la clé la plus proche.
    Si aucune clé trouvée à distance ≤ max_dist -> retourne 5.
    """
    if pd.isna(code):
        return 5   # au lieu de np.nan

    s = str(code)

    # correspondance exacte
    if s in stratum_conv_dict:
        return stratum_conv_dict[s]

    # candidats : même préfixe, sinon toutes les clés
    prefix = get_prefix(s)
    candidates = keys_by_prefix.get(prefix, keys)

    best_dist = None
    best_keys = []

    for k in candidates:
        d = levenshtein(s, k)
        if best_dist is None or d < best_dist:
            best_dist = d
            best_keys = [k]
        elif d == best_dist:
            best_keys.append(k)

    # si rien ou distance trop grande -> catégorie 5
    if best_dist is None or best_dist > max_dist:
        return 5

    # une seule meilleure clé
    if len(best_keys) == 1:
        return stratum_conv_dict[best_keys[0]]

    # plusieurs meilleures clés : prendre la plus fréquente dans df["STRATUM"]
    def freq(k):
        return stratum_freq.get(k, 0)

    best_key = max(best_keys, key=freq)
    return stratum_conv_dict[best_key]

# ------------------------------------------
# 4. application sur les valeurs uniques
# ------------------------------------------
unique_codes = df["STRATUM"].unique()
resolved_map = {code: closest_key(code) for code in unique_codes}

df["STRATUM"] = df["STRATUM"].map(resolved_map)

to_oh.append("STRATUM")

In [22]:
#We add the others cols 
to_oh+=["CNT","CYC"]

In [23]:
#Encoding
from typing import List, Dict

def target_encode(
    df: pd.DataFrame,
    target_col: str,
    cat_cols: List[str],
) -> pd.DataFrame:
    """
    Target encoding simple :
    - fit sur les lignes où target_col n'est pas NaN (train)
    - applique l'encodage à tout le df (train + test)
    - les catégories non vues prennent la moyenne globale du train
    """
    df = df.copy()
    
    # mask train = là où la cible est observée
    mask_train = df[target_col].notna()
    
    # moyenne globale du train
    global_mean = df.loc[mask_train, target_col].mean()
    
    for col in cat_cols:
        # moyenne de la target par catégorie (calculée uniquement sur le train)
        means = (
            df.loc[mask_train]
              .groupby(col, observed=True)[target_col]
              .mean()
        )
        
        df[col] = df[col].map(means).fillna(global_mean)
    
    return df


In [24]:
df = target_encode(df,"MathScore",to_te)
to_rs+=to_te

In [25]:
for c in to_oh: #One Hot pour AG/catboost
    df[c]=df[c].apply(lambda x : f"cat_{str(x)}")

In [26]:
def robust_scale(df, target_col, cols):
    """
    Applique un robust scaling sur les colonnes de `cols` :
    (x - médiane) / IQR, où IQR = Q3 - Q1.
    
    - fit sur tout le df (pas de train/test ici)
    - crée de nouvelles colonnes avec un suffixe (par défaut `_RS`)
    """
    df = df.copy()
    mask_train = df[target_col].notna()
    
    for col in cols:
        q1 = df.loc[mask_train, col].quantile(0.25)
        q2 = df.loc[mask_train, col].quantile(0.50)
        q3 = df.loc[mask_train, col].quantile(0.75)
        iqr = q3 - q1
        
        # éviter division par zéro
        if iqr == 0 or np.isnan(iqr):
            iqr=1
        
        df[col] = (df[col] - q2) / iqr
    
    return df


In [27]:
df = robust_scale(df,"MathScore",to_rs)

In [28]:
df[to_rs].median()

Attempts     0.000000
EFFORT1      0.000000
EFFORT2      0.000000
IMMIG        0.000000
ST003D02T    0.000000
ST003D03T    0.000000
OCOD1        0.001628
OCOD2       -0.000466
OCOD3        0.000000
AGE         -0.000269
GRADE        0.000000
CNTSTUID     0.129832
COBN_S       0.000000
dtype: float64

In [29]:
final_cols+=to_rs
final_cols+=to_oh
final_cols.append("MathScore")

In [30]:
df_final=df[final_cols]

In [ ]:
df_final.to_csv("../../data/df_processed.csv")

In [ ]:
print(to_oh)

['Year', 'STRATUM', 'CNT', 'CYC']
